## CELL 1 - Imports & Logging Setup

In [0]:
import re
import logging
import requests
import pandas as pd
import calendar
from datetime import date, timezone, datetime, timezone
from dateutil.relativedelta import relativedelta

from functools import reduce

from io import StringIO

from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DoubleType

# Set up a logger. Output appears in the cell output panel.
# Change INFO → WARNING to reduce verbosity.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s"
)
logger = logging.getLogger("stl_crime_ingest")

print("✓ Imports loaded")

## Cell 2 - Configuration


In [0]:
# Correct Unity Catalog volume path for this workspace
RAW_OUTPUT_PATH = "/Volumes/workspace/default/raw/crime/city"

START_YEAR  = 2024
START_MONTH = 1
END_YEAR    = datetime.now(timezone.utc).year
END_MONTH   = datetime.now(timezone.utc).month

print(f"✓ Output path: {RAW_OUTPUT_PATH}")
print(f"✓ Date range: {START_YEAR}-{START_MONTH:02d} → {END_YEAR}-{END_MONTH:02d}")

##Cell 3 - Column Alias Map

In [0]:
COLUMN_ALIASES: dict[str, str] = {

    # ── Incident identifier ───────────────────────────────────
    # Pre-2024: "Complaint", "ComplaintNumber", "RptNo", "report_no"
    # 2024+: "IncidentNum"
    "complaint":        "incident_id",
    "complaintnumber":  "incident_id",
    "rptno":            "incident_id",
    "report_no":        "incident_id",
    "incidentnum":      "incident_id",    # ← 2024+ format

    # ── Date and time ─────────────────────────────────────────
    # Pre-2024: "DateOcc", "date_occ", "incident_date"
    # 2024+: "IncidentDate", "OccurredFromTime"
    "dateocc":              "occurred_date",
    "date_occ":             "occurred_date",
    "dateoccurred":         "occurred_date",
    "incident_date":        "occurred_date",
    "incidentdate":         "occurred_date",    # ← 2024+ format
    "timeocc":              "occurred_time",
    "time_occ":             "occurred_time",
    "occurredfromtime":     "occurred_time",    # ← 2024+ format

    # ── Location ──────────────────────────────────────────────
    # Pre-2024: "ILeadsAddress", "address", "location"
    # 2024+: "IncidentLocation", "IntersectionOtherLoc"
    "ileadsaddress":            "address",
    "address":                  "address",
    "location":                 "address",
    "incidentlocation":         "address",         # ← 2024+ format
    "intersectionotherloc":     "address_secondary", # ← 2024+ secondary
    "district":                 "district",
    "neighborhoodname":         "neighborhood",
    "neighborhood_name":        "neighborhood",
    "neighborhood":             "neighborhood",    # ← 2024+ format (exact match)
    "ward":                     "ward",
    "wardno":                   "ward",
    "nbhdnum":                  "ward",            # ← 2024+ neighborhood number
    "xcoord":                   "longitude",
    "x_coord":                  "longitude",
    "longitude":                "longitude",       # ← 2024+ format (exact match)
    "ycoord":                   "latitude",
    "y_coord":                  "latitude",
    "latitude":                 "latitude",        # ← 2024+ format (exact match)

    # ── Crime classification ──────────────────────────────────
    # Pre-2024: "CrimeCode", "Description", "UCR_Type"
    # 2024+: "NIBRS" (code), "Offense" (description),
    #        "NIBRSCategory" (category), "CrimeAgainst" (broad type),
    #        "SRS_UCR" (legacy UCR number)
    "crimecode":            "crime_code",
    "crime_code":           "crime_code",
    "offensecode":          "crime_code",
    "nibrs":                "crime_code",          # ← 2024+ NIBRS offense code
    "description":          "crime_description",
    "crimedescr":           "crime_description",
    "offense":              "crime_description",   # ← 2024+ format
    "incidentnature":       "crime_description",   # ← 2024+ alternate description
    "ucr_type":             "ucr_category",
    "ucrtype":              "ucr_category",
    "nibrscategory":        "ucr_category",        # ← 2024+ NIBRS grouping
    "crimeagainst":         "ucr_category",        # ← 2024+ "Person/Property/Society"
    "srs_ucr":              "ucr_category",        # ← 2024+ legacy UCR number

    # ── Disposition flags ─────────────────────────────────────
    # Pre-2024: "Flag_Crime", "Flag_Unfounded"
    # 2024+: No direct equivalent — "FirearmUsed" and "FelMisdCit" are
    #        new fields we capture but don't have canonical slots for yet.
    #        Mapping FirearmUsed → is_crime is not accurate, so we leave
    #        those as passthroughs and add them to the schema below.
    "flag_crime":       "is_crime",
    "flagcrime":        "is_crime",
    "flag_unfounded":   "is_unfounded",
    "flagunfounded":    "is_unfounded",
    "firearmused":      "firearm_used",    # ← 2024+ new field
    "felmisdcit":       "felony_misd",     # ← 2024+ felony/misdemeanor/citation
    "incidenttpsupr":   "is_supplemented", # ← 2024+ whether report was updated
    "incidentsupplemented": "is_supplemented",
}

print(f"✓ Alias map updated: {len(COLUMN_ALIASES)} known variants")

## Cell 4 - Canonical Schema

In [0]:
CANONICAL_SCHEMA: dict[str, object] = {
    "incident_id":          StringType(),
    "occurred_date":        StringType(),   # raw string — parsed downstream
    "occurred_time":        StringType(),   # raw string — parsed downstream
    "address":              StringType(),
    "address_secondary":    StringType(),   # intersection / cross street (2024+)
    "district":             StringType(),
    "neighborhood":         StringType(),
    "ward":                 StringType(),
    "longitude":            DoubleType(),
    "latitude":             DoubleType(),
    "crime_code":           StringType(),
    "crime_description":    StringType(),
    "ucr_category":         StringType(),
    "is_crime":             StringType(),   # pre-2024 only
    "is_unfounded":         StringType(),   # pre-2024 only
    "firearm_used":         StringType(),   # 2024+ only
    "felony_misd":          StringType(),   # 2024+ only — F/M/C
    "is_supplemented":      StringType(),   # 2024+ only
}

print(f"✓ Canonical schema updated: {len(CANONICAL_SCHEMA)} fields")

## Cell 5 - Build URL Map by Scraping the SLMPD Downloads Page

In [0]:
# ── Date range: last 2 years from today ──────────────────────
TODAY       = date.today()
END_YEAR    = TODAY.year
END_MONTH   = TODAY.month
START_YEAR  = 2024
START_MONTH = 1

SLMPD_BASE = "https://slmpd.org/wp-content/uploads"

# Files confirmed at static /2024/03/ path: Jan, Feb, Mar 2024 (and all older)
# Everything from April 2024 onward uses the dynamic path with 1-month lag.
STATIC_PATH_CUTOFF = (2024, 3)   # last year/month on the static path

def build_url(year: int, month: int) -> str:
    """
    Construct the SLMPD CSV download URL for a given year/month.

    Files up to and including March 2024 live under the static
    /2024/03/ WordPress upload path (all uploaded during site migration).

    April 2024 onward uses the actual upload date as the path, which
    lags the data month by 1 month (e.g. April 2024 data was uploaded
    in May 2024, so path is /2024/05/April2024.csv).

    Examples:
      build_url(2024, 1)  → ".../2024/03/January2024.csv"   ← static
      build_url(2024, 3)  → ".../2024/03/March2024.csv"     ← static
      build_url(2024, 4)  → ".../2024/05/April2024.csv"     ← dynamic
      build_url(2026, 3)  → ".../2026/04/March2026.csv"     ← dynamic
    """
    month_name = calendar.month_name[month]
    filename   = f"{month_name}{year}.csv"

    if (year, month) <= STATIC_PATH_CUTOFF:
        # All files on or before March 2024 use the static migration path
        upload_path = "2024/03"
    else:
        # April 2024+ files use upload date = data date + 1 month
        data_date   = date(year, month, 1)
        upload_date = data_date + relativedelta(months=1)
        upload_path = f"{upload_date.year}/{upload_date.month:02d}"

    return f"{SLMPD_BASE}/{upload_path}/{filename}"


# Preview the boundary URLs to confirm the cutoff looks right
print("Boundary URLs:")
print(f"  Mar 2024 (static): {build_url(2024, 3)}")
print(f"  Apr 2024 (dynamic): {build_url(2024, 4)}")
print(f"  Dec 2024 (dynamic): {build_url(2024, 12)}")
print(f"  Jan 2025 (dynamic): {build_url(2025, 1)}")
print(f"  Mar 2026 (dynamic): {build_url(2026, 3)}")

##Cell 6 - Helper: Column Name Normalizer

In [0]:
def normalize_col_name(raw: str) -> str:
    """
    Clean a raw CSV column header and map it to its canonical name.

    Steps:
      1. Lowercase
      2. Strip leading/trailing whitespace
      3. Remove anything that isn't a letter, digit, or underscore
      4. Look up in COLUMN_ALIASES — return as-is if not found
    """
    # re.sub replaces any character NOT in [a-z0-9_] with nothing.
    # The .lower().strip() handles case and whitespace first.
    cleaned = re.sub(r"[^a-z0-9_]", "", raw.lower().strip())

    # Dict.get(key, default) returns the alias if it exists,
    # or the cleaned name itself if it's an unknown column.
    return COLUMN_ALIASES.get(cleaned, cleaned)


# Inline sanity checks — output prints when you run this cell
test_cases = [
    ("  DateOcc  ",      "occurred_date"),
    ("XCoord",           "longitude"),
    ("Flag_Crime!",      "is_crime"),
    ("NeighborhoodName", "neighborhood"),
    ("some_new_field",   "some_new_field"),  # unknown → passthrough
]
for raw, expected in test_cases:
    result = normalize_col_name(raw)
    status = "✓" if result == expected else "✗"
    print(f"  {status}  normalize_col_name({raw!r:22}) → {result!r}")

## Cell 7 - Helper: HTTP Fetch with Retry

In [0]:
# Headers that mimic a real browser request.
# Without these, slmpd.org returns 403 Forbidden to programmatic requests.
BROWSER_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
}

def fetch_csv_bytes(url: str, timeout: int = 60) -> bytes | None:
    """
    Download a CSV from the given URL, retrying up to 3 times.
    Sends browser-like headers to avoid 403 blocks on slmpd.org.
    Returns raw bytes on success, None if all attempts fail.
    """
    for attempt in range(1, 4):
        try:
            logger.info(f"  GET {url} (attempt {attempt}/3)")
            resp = requests.get(url, headers=BROWSER_HEADERS, timeout=timeout)

            if resp.status_code == 200:
                logger.info(f"  ✓ {len(resp.content):,} bytes received")
                return resp.content

            if resp.status_code == 404:
                logger.warning(f"  404 — not found: {url}")
                return None

            if resp.status_code == 403:
                logger.warning(f"  403 — forbidden on attempt {attempt}/3")
                # No point retrying 403s — it's not a transient error
                return None

            logger.warning(f"  HTTP {resp.status_code} on attempt {attempt}/3")

        except requests.Timeout:
            logger.warning(f"  Timeout on attempt {attempt}/3")
        except requests.RequestException as e:
            logger.warning(f"  Request error on attempt {attempt}/3: {e}")

    logger.error(f"  All attempts failed for {url}")
    return None

print("✓ fetch_csv_bytes() with browser headers")

## Cell 8 - Helper: CSV Parser & Schema Normalizer

In [0]:
def parse_csv_to_df(
    spark: SparkSession,
    raw_bytes: bytes,
    source_label: str
) -> DataFrame | None:
    """
    Parse raw SLMPD CSV bytes into a normalized Spark DataFrame.
    Returns None if the file is empty or unparseable.
    """

    # ── Step 1: Decode bytes → text ──────────────────────────
    try:
        text = raw_bytes.decode("utf-8", errors="replace")
    except Exception as e:
        logger.error(f"Decode error for {source_label}: {e}")
        return None

    # ── Step 2: Parse with Pandas ────────────────────────────
    # dtype=str keeps everything as strings — we cast explicitly
    # in Step 6. Prevents Pandas from auto-guessing types.
    try:
        pdf = pd.read_csv(StringIO(text), dtype=str, low_memory=False)
    except Exception as e:
        logger.error(f"CSV parse error for {source_label}: {e}")
        return None

    if pdf.empty:
        logger.warning(f"Empty CSV for {source_label} — skipping")
        return None

    logger.info(f"  Parsed {len(pdf):,} rows, {len(pdf.columns)} raw columns")

    # ── Step 3: Normalize and de-duplicate column names ──────
    # normalize_col_name() renames each raw header to its canonical
    # name via the alias map. However, multiple source columns can
    # map to the same canonical name — e.g. both "offense" and
    # "incidentnature" → "crime_description" in the 2024+ format.
    # If we don't handle this, Spark ends up with two columns named
    # "crime_description" and throws an AMBIGUOUS_REFERENCE error.
    #
    # Fix: after renaming, detect any duplicates and coalesce them
    # into one column by taking the first non-null value across all
    # columns sharing the same name.

    # Rename all columns using the alias map
    pdf.columns = [normalize_col_name(c) for c in pdf.columns]

    # Detect canonical names that appear more than once
    from collections import Counter
    name_counts = Counter(pdf.columns)
    duplicates  = {name for name, count in name_counts.items() if count > 1}

    if duplicates:
        # The position-based approach gets tricky with pandas internals.
        # Simpler fix: temporarily give each column a unique name using
        # its position, merge the ones that share a canonical name,
        # then we have clean unique columns to work with.

        # Step A: give every column a unique positional name
        pdf.columns = [f"_col_{i}" for i in range(len(pdf.columns))]

        # Step B: we still know the original canonical names from before
        # the rename — rebuild the mapping from position → canonical name
        canonical_names = [normalize_col_name(c) for c in
                          pd.read_csv(StringIO(text), nrows=0).columns]

        # Step C: for each canonical name, find all positions and coalesce
        from collections import defaultdict
        name_to_positions = defaultdict(list)
        for i, name in enumerate(canonical_names):
            name_to_positions[name].append(i)

        # Step D: build the final clean DataFrame column by column
        result_cols = {}
        for canon_name, positions in name_to_positions.items():
            merged = pdf.iloc[:, positions[0]]
            for pos in positions[1:]:
                merged = merged.combine_first(pdf.iloc[:, pos])
            result_cols[canon_name] = merged

        pdf = pd.DataFrame(result_cols)
        logger.info(f"  Resolved duplicate columns: {duplicates}")

    # ── Step 4: Add missing canonical columns as null ────────
    # Guarantees every output DataFrame has all expected columns
    # even if this file vintage didn't include them.
    for col in CANONICAL_SCHEMA:
        if col not in pdf.columns:
            pdf[col] = None

    # ── Step 5: Drop non-canonical columns ───────────────────
    # Strips any fields not in CANONICAL_SCHEMA.
    pdf = pdf[[col for col in CANONICAL_SCHEMA if col in pdf.columns]]

    # ── Step 6: Convert to Spark + cast types ────────────────
    # Spark's cast turns unparseable values (e.g. a non-numeric
    # latitude) into null rather than raising an error.
    sdf = spark.createDataFrame(pdf)
    for col_name, spark_type in CANONICAL_SCHEMA.items():
        if col_name in sdf.columns:
            sdf = sdf.withColumn(col_name, F.col(col_name).cast(spark_type))

    # ── Step 7: Attach provenance metadata ───────────────────
    # _source traces any row back to the exact file it came from.
    # _ingested_at records when this pipeline run occurred.
    from datetime import timezone
    sdf = sdf.withColumn("_source", F.lit(source_label))
    sdf = sdf.withColumn("_ingested_at", F.lit(datetime.now(timezone.utc).isoformat()))

    return sdf


print("✓ parse_csv_to_df() defined")

##Cell 9 - Single Month Fetch + Parse

In [0]:
def fetch_month(spark: SparkSession, year: int, month: int) -> DataFrame | None:
    """
    Download and parse one month of SLMPD crime data.
    URL is constructed directly from build_url() — no scraping needed.
    Returns a normalized Spark DataFrame, or None if unavailable.
    """
    label = f"slmpd_{year}_{month:02d}"
    url   = build_url(year, month)

    logger.info(f"Fetching {label} from {url}")
    raw = fetch_csv_bytes(url)

    if raw is None:
        return None

    sdf = parse_csv_to_df(spark, raw, source_label=label)
    if sdf is None:
        return None

    return (
        sdf
        .withColumn("year",  F.lit(year).cast("int"))
        .withColumn("month", F.lit(month).cast("int"))
    )


print("✓ fetch_month() defined")

## Cell 10 — Main Ingestion Loop

In [0]:
current_dt    = date.today()
success_count = 0
skip_count    = 0
all_frames: list[DataFrame] = []

print(f"Starting ingestion: {START_YEAR}-{START_MONTH:02d} → {END_YEAR}-{END_MONTH:02d}")
print(f"Output path: {RAW_OUTPUT_PATH}")
print("=" * 60)

for year in range(START_YEAR, END_YEAR + 1):
    for month in range(1, 13):

        # ── Skip months outside our 2-year window ────────────
        # Lower bound: don't go earlier than START_YEAR/START_MONTH
        if year == START_YEAR and month < START_MONTH:
            continue

        # Upper bound: don't request months that haven't happened yet
        if year == END_YEAR and month > END_MONTH:
            break

        print(f"\n{'─' * 40}")
        print(f"Processing {year}-{month:02d}...")

        # ── Fetch and parse this month ────────────────────────
        sdf = fetch_month(spark, year, month)

        if sdf is None:
            print(f"  ⚠ Skipped {year}-{month:02d} (no data returned)")
            skip_count += 1
            continue

        # ── Write partition to Parquet immediately ────────────
        # Path follows Hive partitioning convention so Spark can
        # read the full dataset later with automatic partition pruning.
        partition_path = f"{RAW_OUTPUT_PATH}/year={year}/month={month:02d}"
        sdf.write.mode("overwrite").parquet(partition_path)

        row_count = sdf.count()
        print(f"  ✓ Wrote {row_count:,} rows → {partition_path}")
        success_count += 1
        all_frames.append(sdf)

# ── Summary ───────────────────────────────────────────────────
print(f"\n{'=' * 60}")
print(f"Ingestion complete:")
print(f"  ✓ Months written:  {success_count}")
print(f"  ⚠ Months skipped:  {skip_count}")

if not all_frames:
    raise RuntimeError(
        "No data was ingested. "
        "Check network access and verify the date range in Cell 3."
    )

In [0]:
# Union all monthly DataFrames into one combined dataset.
# reduce() applies DataFrame.union() repeatedly across the list:
#   frame1.union(frame2).union(frame3)... and so on
combined_df = reduce(DataFrame.union, all_frames)

total_rows = combined_df.count()
print(f"✓ Combined DataFrame: {total_rows:,} total rows across {success_count} months")
print(f"\nSchema:")
combined_df.printSchema()

## Cell 11 - Dataframe Litmus Tests

In [0]:
print("=== NULL RATE BY COLUMN (%) ===")
# For each canonical column, calculate what % of rows are null.
# A well-ingested month should have near-zero nulls on the key
# identifier and date fields.
null_rates = combined_df.select([
    F.round(
        F.count(F.when(F.col(c).isNull(), c)) / F.count(F.lit(1)) * 100, 2
    ).alias(c)
    for c in CANONICAL_SCHEMA.keys()
])
display(null_rates)


print("=== ROW COUNT BY YEAR / MONTH ===")
display(
    combined_df
    .groupBy("year", "month")
    .count()
    .orderBy("year", "month")
)

print("=== UCR CATEGORY DISTRIBUTION ===")
display(
    combined_df
    .groupBy("ucr_category")
    .count()
    .orderBy("count", ascending=False)
)


print("=== TOP 20 NEIGHBORHOODS BY RECORD COUNT ===")
display(
    combined_df
    .groupBy("neighborhood")
    .count()
    .orderBy("count", ascending=False)
    .limit(20)
)

## Cell 14 — Read Back from Parquet

In [0]:
persisted_df    = spark.read.parquet(RAW_OUTPUT_PATH)
persisted_count = persisted_df.count()

print(f"Rows read back from Parquet: {persisted_count:,}")
print(f"Distinct partitions found:   {persisted_df.select('year', 'month').distinct().count()}")

if persisted_count == total_rows:
    print("✓ Row count matches in-memory DataFrame — write verified")
else:
    print(f"⚠ Mismatch: in-memory={total_rows:,}, on-disk={persisted_count:,}")